# 🍔 McDonald's Menu Nutrition Analysis

**Author:** Baby  
**Dataset:** [McDonald's Menu – Kaggle](https://www.kaggle.com/datasets/mcdonalds/nutrition-facts)  
**Tools:** Python · Pandas · Matplotlib · Seaborn · NumPy

---

## Project Overview

This project performs an end-to-end Exploratory Data Analysis (EDA) on McDonald's full menu nutritional data.  
The dataset contains **260 menu items** across **9 food categories** with **24 nutritional attributes** per item.

### Questions I'm answering:
1. Which food categories are the most calorie-dense?
2. What are the most extreme items by calories, sodium, and sugar?
3. How do macronutrients (protein, fat, carbs) vary across categories?
4. Which nutrients are most strongly correlated with calorie count?
5. How much of the recommended daily intake does a typical McDonald's meal contribute?

---

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import math

warnings.filterwarnings('ignore')

# Plot styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

print("Libraries loaded successfully.")

## 2. Load the Dataset

In [ ]:
df = pd.read_csv('data/menu.csv')
print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

## 3. Data Overview & Quality Check

Before diving into analysis, we need to understand the structure and check for any data quality issues — missing values, incorrect types, or unexpected values.

In [ ]:
# Column data types
print("=== Data Types ===")
print(df.dtypes)
print()

# Missing values
print("=== Missing Values ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "No missing values found. ✓")

In [ ]:
# Statistical summary of all numeric columns
df.describe().T.style.background_gradient(cmap='YlOrRd', subset=['mean', 'max'])

In [ ]:
# Category breakdown
print("=== Items per Category ===")
cat_counts = df['Category'].value_counts()
print(cat_counts)

**Observations:**
- The dataset is clean — **no missing values** across all 24 columns.
- `Serving Size` is stored as a string (e.g., "4.1 oz (116 g)") — it's not used in numeric analysis but documents real-world portion context.
- `Coffee & Tea` dominates the menu with **95 items (36.5%)**, largely due to size variations of the same drink.
- `Salads` and `Desserts` are the smallest categories (6 and 7 items respectively).
- Calorie values range from **0** (Diet sodas, water, black coffee) to **1,880** — a wide spread that warrants deeper exploration.

## 4. Calorie Distribution

Calories are the most commonly referenced nutritional metric. Let's understand how they are distributed across the entire menu.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['Calories'], bins=30, color='#E63946', edgecolor='white', linewidth=0.7)
axes[0].axvline(df['Calories'].mean(), color='navy', linestyle='--', linewidth=1.8, label=f"Mean: {df['Calories'].mean():.0f}")
axes[0].axvline(df['Calories'].median(), color='gold', linestyle='--', linewidth=1.8, label=f"Median: {df['Calories'].median():.0f}")
axes[0].set_title('Distribution of Calories Across All Menu Items', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Calories')
axes[0].set_ylabel('Count of Items')
axes[0].legend()

# Boxplot by category
cat_order = df.groupby('Category')['Calories'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='Calories', y='Category', order=cat_order,
            palette='RdYlGn_r', ax=axes[1])
axes[1].set_title('Calorie Range by Category', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Calories')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('visuals/01_calorie_distribution.png', bbox_inches='tight')
plt.show()

**Observations:**
- The calorie distribution is **right-skewed** — most items cluster between 100–500 kcal, with a long tail of high-calorie items.
- The mean (368 kcal) is pulled above the median (340 kcal) by those extreme high-calorie outliers.
- **Chicken & Fish** and **Smoothies & Shakes** have the widest calorie ranges, driven by portion size variation.
- **Beverages** are the lowest-calorie category — most are zero-calorie or low-sugar drinks.

## 5. Category-Level Calorie Analysis

Which food categories contribute the most calories on average? This reveals where the "hidden calories" in the McDonald's menu are.

In [ ]:
cat_cal = df.groupby('Category')['Calories'].mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d62728' if v > 500 else '#ff7f0e' if v > 300 else '#2ca02c' for v in cat_cal.values]
bars = ax.barh(cat_cal.index, cat_cal.values, color=colors, edgecolor='white', height=0.6)

# Add value labels
for bar, val in zip(bars, cat_cal.values):
    ax.text(val + 8, bar.get_y() + bar.get_height()/2,
            f'{val:.0f} kcal', va='center', fontsize=10)

ax.axvline(2000/3, color='gray', linestyle=':', linewidth=1.5, label='⅓ of 2000 kcal daily intake')
ax.set_title('Average Calories per Category', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Average Calories (kcal)')
ax.set_xlim(0, 720)
ax.legend(loc='lower right')
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('visuals/02_avg_calories_by_category.png', bbox_inches='tight')
plt.show()

**Observations:**
- **Chicken & Fish** tops the chart at an average of **553 kcal** per item — higher even than Beef & Pork (494 kcal), likely due to breading and frying.
- **Smoothies & Shakes** average **531 kcal** — a category often perceived as a healthier choice but actually one of the most calorie-dense.
- A single Breakfast item averages **527 kcal**, meaning a typical McDonald's breakfast already consumes ~26% of a 2000-kcal daily intake.
- **Salads** average only 270 kcal — but dressings and toppings not always reflected can push this significantly higher.

## 6. Top Extreme Items — Calories, Sodium & Sugar

Identifying the outliers is just as important as understanding averages. These are the items a nutritionally-conscious consumer should be most aware of.

In [ ]:
def plot_top_items(metric, title, color, ax, n=10):
    top = df.nlargest(n, metric)[['Item', 'Category', metric]].reset_index(drop=True)
    # Truncate long names
    top['Short Name'] = top['Item'].str[:40] + top['Item'].apply(lambda x: '...' if len(x) > 40 else '')
    bars = ax.barh(top['Short Name'][::-1], top[metric][::-1], color=color, edgecolor='white', height=0.65)
    for bar, val in zip(bars, top[metric][::-1]):
        ax.text(val + top[metric].max()*0.01, bar.get_y() + bar.get_height()/2,
                f'{val}', va='center', fontsize=9)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

plot_top_items('Calories', 'Top 10 Highest Calorie Items (kcal)', '#d62728', axes[0])
plot_top_items('Sodium', 'Top 10 Highest Sodium Items (mg)', '#e07b39', axes[1])
plot_top_items('Sugars', 'Top 10 Highest Sugar Items (g)', '#9467bd', axes[2])

plt.suptitle('Extreme Nutritional Outliers Across the Menu', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('visuals/03_top_extreme_items.png', bbox_inches='tight')
plt.show()

**Observations:**
- The **40-piece Chicken McNuggets** is the single most calorie-dense item at **1,880 kcal** — nearly a full day's recommended intake in one order. It also holds the sodium record at **3,600 mg** (156% of the 2,300 mg daily limit).
- The top 4 highest-calorie items (1,000+ kcal) all belong to the **Big Breakfast with Hotcakes** family — a caution for morning meal choices.
- The **sugar ranking is dominated by Smoothies & Shakes** — 9 of the top 10 highest-sugar items. The Large Chocolate Shake contains **128g of sugar**, equivalent to ~32 teaspoons.
- This is a **clear pattern**: beverages (shakes, frappes) are the biggest sugar risk, while food items (nuggets, breakfast combos) drive sodium and calorie extremes.

## 7. Macronutrient Breakdown by Category

Beyond calories, understanding the protein–fat–carbohydrate balance across categories reveals the nutritional character of each food group.

In [ ]:
macros = df.groupby('Category')[['Protein', 'Total Fat', 'Carbohydrates']].mean().round(1)
macros = macros.sort_values('Protein', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(macros))
width = 0.25

b1 = ax.bar(x - width, macros['Protein'], width, label='Protein (g)', color='#2196F3', edgecolor='white')
b2 = ax.bar(x, macros['Total Fat'], width, label='Total Fat (g)', color='#F44336', edgecolor='white')
b3 = ax.bar(x + width, macros['Carbohydrates'], width, label='Carbohydrates (g)', color='#FF9800', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(macros.index, rotation=30, ha='right')
ax.set_title('Average Macronutrient Profile by Category', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Grams per Item')
ax.legend()
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('visuals/04_macronutrient_by_category.png', bbox_inches='tight')
plt.show()

print(macros)

**Observations:**
- **Chicken & Fish** and **Beef & Pork** lead in protein (29g and 27g respectively) — the best categories for protein-to-calorie value.
- **Smoothies & Shakes** have a dramatically carbohydrate-heavy profile (avg ~96g carbs) with low protein — a poor macro balance for satiety.
- **Coffee & Tea** shows surprisingly high fat (avg ~11g) and carbs (~37g) — driven by specialty drinks like Frappes and McCafé lattes with added syrups and cream.
- **Salads** offer the best macro balance relative to calories: moderate protein (~20g), lower fat, and reasonable carbs.

## 8. Sodium Analysis — The Hidden Health Risk

The WHO recommends under 2,000 mg of sodium per day. Let's see how McDonald's categories stack up — sodium is often overlooked compared to calories.

In [ ]:
cat_sodium = df.groupby('Category')['Sodium'].mean().sort_values(ascending=False)
who_limit = 2000
pct_of_limit = (cat_sodium / who_limit * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart — avg sodium
colors_s = ['#d32f2f' if v > 1000 else '#f57c00' if v > 500 else '#388e3c' for v in cat_sodium.values]
axes[0].barh(cat_sodium.index[::-1], cat_sodium.values[::-1], color=colors_s[::-1], edgecolor='white', height=0.6)
axes[0].axvline(who_limit, color='crimson', linestyle='--', linewidth=1.8, label='WHO Daily Limit (2000mg)')
axes[0].set_title('Average Sodium per Category (mg)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Sodium (mg)')
axes[0].legend()
axes[0].spines[['top','right']].set_visible(False)

# % of daily limit
axes[1].barh(pct_of_limit.index[::-1], pct_of_limit.values[::-1], color=colors_s[::-1], edgecolor='white', height=0.6)
axes[1].axvline(100, color='crimson', linestyle='--', linewidth=1.8, label='100% of Daily Limit')
axes[1].set_title('Avg Sodium as % of WHO Daily Limit', fontsize=12, fontweight='bold')
axes[1].set_xlabel('% of Daily Limit')
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter())
axes[1].legend()
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('visuals/05_sodium_analysis.png', bbox_inches='tight')
plt.show()

**Observations:**
- **Chicken & Fish** averages 1,258 mg of sodium per item — over **60% of the WHO daily limit** in a single dish.
- **Breakfast** follows at 1,211 mg — meaning a typical McDonald's breakfast accounts for more than half the recommended daily sodium intake.
- **Beef & Pork** averages 1,021 mg (51% of the daily limit).
- The 40-piece Chicken McNuggets at 3,600 mg sodium represents **180% of the WHO daily limit** — in a single item.
- **Beverages** (41 mg avg) and **Desserts** (117 mg avg) are comparatively low, making them the safest sodium choices.

## 9. Correlation Analysis — What Drives Calorie Count?

Understanding which nutrients are most predictive of calories helps model the nutritional structure of the menu.

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
corr_matrix = numeric_df.corr()

# Full heatmap
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, ax=axes[0], annot_kws={'size': 7},
            vmin=-1, vmax=1, center=0, cbar_kws={'shrink': 0.7})
axes[0].set_title('Full Correlation Matrix', fontsize=13, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45, labelsize=8)
axes[0].tick_params(axis='y', rotation=0, labelsize=8)

# Calorie correlations only
cal_corr = corr_matrix['Calories'].drop('Calories').sort_values()
colors_c = ['#d62728' if v > 0 else '#1f77b4' for v in cal_corr.values]
axes[1].barh(cal_corr.index, cal_corr.values, color=colors_c, edgecolor='white', height=0.65)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Correlation of Each Nutrient with Calories', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Pearson Correlation Coefficient')
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('visuals/06_correlation_analysis.png', bbox_inches='tight')
plt.show()

**Observations:**
- **Total Fat** (r = 0.904) and **Calories from Fat** (r = 0.905) are near-perfectly correlated with total calories — fat is the dominant calorie driver on this menu.
- **Protein** (r = 0.788) and **Carbohydrates** (r = 0.782) also show strong positive correlation with calories, as expected from macronutrient caloric contribution.
- **Sugars** (r = 0.260) have a surprisingly *weak* correlation with calories — because many high-sugar items (shakes, frappes) are also high-fat, while diet drinks have zero calories and zero sugar.
- **Vitamin C** is the only nutrient with a slight *negative* correlation with calories (r = −0.069), likely because it's concentrated in low-calorie beverages and salads.
- The strong cross-correlations between fat metrics (Total Fat, Saturated Fat, Cholesterol) confirm multicollinearity — these should not all be used together in a predictive model.

## 10. Daily Value % Analysis — How Much of Your Day Does One Item Use?

The FDA's % Daily Value (%DV) is based on a 2,000 kcal diet. Values above 20% are considered "high". Let's see which nutrients McDonald's items consistently exceed.

In [ ]:
dv_cols = [c for c in df.columns if '% Daily Value' in c]
dv_avg = df[dv_cols].mean().round(1).sort_values(ascending=False)
dv_labels = [c.replace(' (% Daily Value)', '') for c in dv_avg.index]

fig, ax = plt.subplots(figsize=(10, 6))
bar_colors = ['#d62728' if v >= 20 else '#ff7f0e' if v >= 10 else '#2ca02c' for v in dv_avg.values]
bars = ax.barh(dv_labels[::-1], dv_avg.values[::-1], color=bar_colors[::-1], edgecolor='white', height=0.6)

ax.axvline(20, color='crimson', linestyle='--', linewidth=1.8, label='20% DV threshold ("High")')
ax.axvline(5, color='steelblue', linestyle=':', linewidth=1.5, label='5% DV threshold ("Low")')

for bar, val in zip(bars, dv_avg.values[::-1]):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val}%', va='center', fontsize=9)

ax.set_title('Average % Daily Value per Nutrient (All Menu Items)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Average % Daily Value')
ax.legend()
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('visuals/07_daily_value_pct.png', bbox_inches='tight')
plt.show()

**Observations:**
- **Saturated Fat** is the most concerning nutrient at 30% DV on average — exceeding the "High" threshold of 20%.
- **Total Fat, Calcium, Sodium, and Cholesterol** all average above 20% DV, meaning a typical McDonald's item is "high" in all four of these nutrients.
- **Dietary Fiber** averages only **6.5% DV** — a fast food diet is extremely low in fiber, which is important for digestive health.
- **Vitamin C** averages just 8.5% DV — unsurprising given the limited fresh produce on the menu.

## 11. Trans Fat & Zero-Calorie Items

Two edge cases worth examining: items with trans fat (linked to cardiovascular disease) and zero-calorie items for those seeking guilt-free options.

In [ ]:
# Trans fat items
trans_items = df[df['Trans Fat'] > 0][['Category', 'Item', 'Trans Fat']].sort_values('Trans Fat', ascending=False)
print(f"Items containing trans fat: {len(trans_items)}")
print()
print(trans_items.to_string(index=False))

In [ ]:
# Zero calorie items
zero_cal = df[df['Calories'] == 0][['Category', 'Item']]
print(f"Zero-calorie items: {len(zero_cal)}")
print()
print(zero_cal.to_string(index=False))

**Observations:**
- **10 menu items** contain trans fat, all from Beef & Pork, Breakfast, and Coffee & Tea categories.
- The **Double Quarter Pounder with Cheese** has the highest trans fat at 2.5g per serving.
- **16 items are zero-calorie**, all from the Beverages and Coffee & Tea categories (diet sodas, water, black iced tea, plain coffee).
- Zero-calorie options are available but limited — and several (plain coffee, unsweetened tea) lose their calorie-free status when customized with syrups or cream.

## 12. Distribution Overview — All Numeric Columns

In [ ]:
def plot_column_distributions(df, n_shown=12, n_per_row=4):
    """
    Plot distribution histograms or bar charts for numeric columns.
    """
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    cols_to_plot = numeric_cols[:n_shown]
    n_rows = math.ceil(len(cols_to_plot) / n_per_row)

    fig, axes = plt.subplots(n_rows, n_per_row, figsize=(5 * n_per_row, 4 * n_rows))
    axes = axes.flatten()

    for i, col in enumerate(cols_to_plot):
        axes[i].hist(df[col].dropna(), bins=25, color='#457b9d', edgecolor='white', linewidth=0.5)
        axes[i].set_title(col, fontsize=9, fontweight='bold')
        axes[i].set_ylabel('Count')
        axes[i].tick_params(labelsize=8)
        axes[i].spines[['top','right']].set_visible(False)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Distribution of All Numeric Nutritional Columns', fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('visuals/08_column_distributions.png', bbox_inches='tight')
    plt.show()

plot_column_distributions(df, n_shown=12, n_per_row=4)

## 13. Summary & Key Findings

---

### 🔑 Key Findings

| # | Finding |
|---|---|
| 1 | **Chicken & Fish is the highest-calorie category** (avg 553 kcal), surpassing even Beef & Pork |
| 2 | **Smoothies & Shakes are deceptively unhealthy** — avg 531 kcal and 9 of the top 10 sugar-heavy items |
| 3 | **The 40-piece McNuggets is the most extreme item** — 1,880 kcal and 3,600 mg sodium (156% of daily limit) |
| 4 | **Total Fat is the strongest calorie predictor** (r = 0.904), not sugar (r = 0.260) |
| 5 | **Saturated Fat averages 30% Daily Value** across all menu items — consistently exceeding the "high" threshold |
| 6 | **Breakfast items average 1,211 mg sodium** — over 60% of WHO's 2,000 mg daily recommendation per item |
| 7 | **Only 16 items are zero-calorie** — all beverages; the menu offers limited truly low-calorie food options |
| 8 | **Salads offer the best overall nutritional balance**: moderate protein, lower sodium, reasonable calories |

---

### 📌 If You're Eating at McDonald's
- **Lowest risk:** Beverages (plain), Salads (without dressing), black Coffee/Tea
- **Watch out for:** Smoothies & Shakes (sugar), Breakfast combos and Chicken items (sodium), anything "Large"
- **Best protein-per-calorie:** Salads and grilled Chicken items

---

### 🔭 Potential Next Steps
- Build a **calorie prediction model** using fat, protein, and carbs as features (strong correlation foundation exists)
- Cluster menu items using **K-Means** to identify natural nutritional groupings
- Perform **serving-size normalization** to compare items on a per-100g basis
- Analyse **time-of-day nutrition** if ordering data were available